# 05 — Modeling: Baseline Classifiers

Trains Logistic Regression, Naïve Bayes, k-Nearest Neighbours, and Decision Tree with `RandomizedSearchCV` (TimeSeriesSplit = 5). Evaluates each on the held-out test set and saves models to disk.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import json, joblib
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

RANDOM_STATE = 42
os.makedirs('../models', exist_ok=True)
os.makedirs('../figures', exist_ok=True)


## 1 · Load Features and Split

In [ ]:
import features, splits

df = pd.read_parquet('../data/modeling_table.parquet')
train_mask, val_mask, test_mask = splits.temporal_split(df, train_frac=0.60, val_frac=0.20)

X, feature_names = features.build_feature_matrix(df, train_mask)
y = df['y_primary'].values

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"Positive rate — Train: {y_train.mean():.3f}  Val: {y_val.mean():.3f}  Test: {y_test.mean():.3f}")


## 2 · Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# Combined train+val for final fit
X_trainval   = np.vstack([X_train, X_val])
X_trainval_s = scaler.transform(X_trainval)
y_trainval   = np.concatenate([y_train, y_val])


## 3 · Evaluation Helper

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score, brier_score_loss,
    roc_curve,
)

results = {}

def evaluate(name, model, X_te, y_te, proba=True):
    if proba:
        probs = model.predict_proba(X_te)[:, 1]
    else:
        probs = model.decision_function(X_te)
    preds = model.predict(X_te)
    metrics = {
        'ROC-AUC' : round(roc_auc_score(y_te, probs), 4),
        'PR-AUC'  : round(average_precision_score(y_te, probs), 4),
        'Accuracy': round(accuracy_score(y_te, preds), 4),
        'F1'      : round(f1_score(y_te, preds), 4),
        'Brier'   : round(brier_score_loss(y_te, probs), 4),
    }
    results[name] = metrics
    for k, v in metrics.items():
        print(f"  {k:9s}: {v}")
    return probs


## 4 · Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import loguniform

tscv = TimeSeriesSplit(n_splits=5)

lr_params = {
    'C'       : loguniform(1e-3, 1e2),
    'penalty' : ['l1', 'l2'],
    'solver'  : ['saga'],
    'max_iter': [1000],
}

lr_search = RandomizedSearchCV(
    LogisticRegression(random_state=RANDOM_STATE),
    lr_params, n_iter=20, cv=tscv,
    scoring='roc_auc', n_jobs=-1,
    random_state=RANDOM_STATE, verbose=0,
)
lr_search.fit(X_train_s, y_train)
print("Best params:", lr_search.best_params_)

lr_best = lr_search.best_estimator_
lr_best.fit(X_trainval_s, y_trainval)

print("\nTest-set performance:")
lr_probs = evaluate('LogReg', lr_best, X_test_s, y_test)

joblib.dump(lr_best, '../models/logreg.pkl')
joblib.dump(scaler,  '../models/standard_scaler.pkl')
print("Saved → ../models/logreg.pkl")


## 5 · Naïve Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb_params = {'var_smoothing': loguniform(1e-12, 1e-3)}

nb_search = RandomizedSearchCV(
    GaussianNB(), nb_params, n_iter=20, cv=tscv,
    scoring='roc_auc', n_jobs=-1,
    random_state=RANDOM_STATE, verbose=0,
)
nb_search.fit(X_train_s, y_train)
print("Best params:", nb_search.best_params_)

nb_best = nb_search.best_estimator_
nb_best.fit(X_trainval_s, y_trainval)

print("\nTest-set performance:")
nb_probs = evaluate('NaiveBayes', nb_best, X_test_s, y_test)

joblib.dump(nb_best, '../models/naive_bayes.pkl')
print("Saved → ../models/naive_bayes.pkl")


## 6 · k-Nearest Neighbours

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_params = {
    'n_neighbors': [5, 11, 21, 51, 101],
    'weights'    : ['uniform', 'distance'],
    'metric'     : ['euclidean', 'manhattan'],
}

knn_search = RandomizedSearchCV(
    KNeighborsClassifier(), knn_params, n_iter=15, cv=tscv,
    scoring='roc_auc', n_jobs=-1,
    random_state=RANDOM_STATE, verbose=0,
)
knn_search.fit(X_train_s, y_train)
print("Best params:", knn_search.best_params_)

knn_best = knn_search.best_estimator_
knn_best.fit(X_trainval_s, y_trainval)

print("\nTest-set performance:")
knn_probs = evaluate('kNN', knn_best, X_test_s, y_test)

joblib.dump(knn_best, '../models/knn.pkl')
print("Saved → ../models/knn.pkl")


## 7 · Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_params = {
    'max_depth'       : [3, 5, 8, 12, None],
    'min_samples_leaf': [10, 25, 50, 100],
    'max_features'    : ['sqrt', 'log2', None],
    'criterion'       : ['gini', 'entropy'],
}

dt_search = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    dt_params, n_iter=20, cv=tscv,
    scoring='roc_auc', n_jobs=-1,
    random_state=RANDOM_STATE, verbose=0,
)
dt_search.fit(X_train, y_train)
print("Best params:", dt_search.best_params_)

dt_best = dt_search.best_estimator_
dt_best.fit(X_trainval, y_trainval)

print("\nTest-set performance:")
dt_probs = evaluate('DecisionTree', dt_best, X_test, y_test)

joblib.dump(dt_best, '../models/decision_tree.pkl')
print("Saved → ../models/decision_tree.pkl")


## 8 · ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, probs in [
    ('LogReg', lr_probs),
    ('NaiveBayes', nb_probs),
    ('kNN', knn_probs),
    ('DecisionTree', dt_probs),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = results[name]['ROC-AUC']
    ax.plot(fpr, tpr, label=f"{name}  (AUC = {auc:.4f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Baseline Models (Test Set)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../figures/05_roc_curves_baselines.png', bbox_inches='tight')
plt.show()


## 9 · Comparison Table — Baseline Models

In [ ]:
comparison = pd.DataFrame(results).T
comparison.index.name = 'Model'
print(comparison.to_string())
comparison
